<a href="https://colab.research.google.com/github/evandwh/ST554_Homework/blob/main/Homework_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment: Homework #7

Author: Evan Whitfield

Course: ST554 - Spring 2026

### Data

We will use a dataset from the UCI Machine Learning Repository. This data set is about wine quality. You can learn more about the data here.

The data description describes the following variables:
**Input variables** (based on physicochemical tests)

1- fixed acidity

2- volatile acidity

3- citric acid

4- residual sugar

5- chlorides

6- free sulfur dioxide

7- total sulfur dioxide

8- density

9- pH

10- sulphates

11- alcohol

**Output variable** (based on sensory data):
12- quality (score between 0 and 10

## Our purpose

Rather than try to predict quality, we will make our target variable for fitting multiple linear regression type models `alcohol`.


For fitting logistic regression type models we’ll use the `type of wine` as the response variable.

## Read In and Combine Data

First, we will read in the CSV files into seperate dataframes. We will also create a new column for the type of wine, so that when we combine the data sets, we will know which type of wine each row is from.

In [7]:
import pandas as pd

red_wine_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Colab Data/winequality-red.csv', sep=';')
red_wine_data['type'] = 'red'

white_wine_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Colab Data/winequality-white.csv', sep=';')
white_wine_data['type'] = 'white'

In [8]:
red_wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


In [9]:
white_wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,white
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,white
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,white
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white


Both datasets were read into Python correctly.

Now we will combine the dataframes into one master dataframe called `wine_data`.

In [10]:
wine_data = pd.concat([red_wine_data, white_wine_data])
wine_data["type_num"] = wine_data["type"].map({"red": 0, "white": 1})
wine_data.head(-5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.4,0.700,0.00,1.90,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red,0
1,7.8,0.880,0.00,2.60,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red,0
2,7.8,0.760,0.04,2.30,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red,0
3,11.2,0.280,0.56,1.90,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red,0
4,7.4,0.700,0.00,1.90,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4888,6.8,0.220,0.36,1.20,0.052,38.0,127.0,0.99330,3.04,0.54,9.2,5,white,1
4889,4.9,0.235,0.27,11.75,0.030,34.0,118.0,0.99540,3.07,0.50,9.4,6,white,1
4890,6.1,0.340,0.29,2.20,0.036,25.0,100.0,0.98938,3.06,0.44,11.8,6,white,1
4891,5.7,0.210,0.32,0.90,0.038,38.0,121.0,0.99074,3.24,0.46,10.6,6,white,1


## Split the Data

We are going to split up the data set into a training and test set. For this, we will use stratified sampling to
make sure that you have a similar proportion of white and red wines in the training and test sets.

In [11]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

X = wine_data.drop(columns=["alcohol"])
y = wine_data["alcohol"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = wine_data['type'])

In [12]:
print(X_train.head())

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
2511            6.2              0.30         0.49            11.2      0.058   
1463            8.3              0.19         0.49             1.2      0.051   
3637            8.7              0.30         0.34             4.8      0.018   
497             7.2              0.34         0.32             2.5      0.090   
826             7.5              0.27         0.34             2.3      0.050   

      free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
2511                 68.0                 215.0  0.99656  3.19       0.60   
1463                 11.0                 137.0  0.99180  3.06       0.46   
3637                 23.0                 127.0  0.99474  3.12       0.49   
497                  43.0                 113.0  0.99660  3.32       0.79   
826                   4.0                   8.0  0.99510  3.40       0.64   

      quality   type  type_num  
2511        6  wh

In [13]:
print(X_test.head())

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
140             6.3              0.31         0.34             2.2      0.045   
1246            8.0              0.20         0.40             5.2      0.055   
3683            6.8              0.15         0.41            12.9      0.044   
4223            5.3              0.43         0.11             1.1      0.029   
3638            6.4              0.29         0.32             2.4      0.014   

      free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
140                  20.0                  77.0  0.99270  3.30       0.43   
1246                 41.0                 167.0  0.99530  3.18       0.40   
3683                 79.5                 183.0  0.99742  3.24       0.78   
4223                  6.0                  51.0  0.99076  3.51       0.48   
3638                 34.0                  89.0  0.99008  3.24       0.66   

      quality   type  type_num  
140         5  wh

## Regression Models

We will create a set for y_train and y_test for the .  x_train and x_test will change based on the model/variables we choose for each task.

### 1 - MLR Models

In [14]:
X_train_reg = X_train.drop(columns = ['type'])
X_test_reg = X_test.drop(columns = ['type'])

x_means = X_train_reg.apply(np.mean, axis=0)
x_stds = X_train_reg.apply(np.std, axis=0)

#Standardizing
X_train_reg = X_train_reg.apply(lambda x: (x - np.mean(x)) / np.std(x), axis=0)

### Linear Full Model

Using a normal linear model without removing any of the variables.

In [71]:
lr_full = LinearRegression()
lr_full.fit(X_train_reg, y_train)

cv_full_model = cross_validate(
    lr_full,
    X_train_reg,
    y_train,
    cv=5,
    scoring = "neg_mean_squared_error"
)

### Interaction Terms

We still will not remove any variables, but will now introduce interation terms between the variables.

In [74]:
# Add interaction terms
poly_interation = PolynomialFeatures(
    degree=2,
    interaction_only=True,
    include_bias=False
)
X_interaction = poly_interation.fit_transform(X_train_reg)

inter_model = LinearRegression()
inter_model.fit(X_interaction, y_train)

# Cross-validate
cv_interaction_model = cross_validate(
    LinearRegression(),
    X_interaction,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

### Polynomial Terms

Just because I want to see what happens, I am going to (most likely) over train our model with degree 2 terms with interaction_only = False. I want the model to square some terms to see what happens. Most likely, some of the variables need to be removed, but lets give it a shot first!

In [78]:
# Add interaction terms
poly_ = PolynomialFeatures(
    degree=2,
    interaction_only=False,
    include_bias=False
)
X_poly = poly_.fit_transform(X_train_reg)

poly_model = LinearRegression()
poly_model.fit(X_poly, y_train)

# Cross-validate
cv_poly_model = cross_validate(
    poly_model,
    X_poly,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

### Best Guess

After doing some exploring on the internet, it seems like density and sugar are the most important factors in determining alcohol level, so we will use those only in our last prediction model.

In [80]:
X_best_maybe = poly_interation.fit_transform(X_train_reg[["residual sugar", "density"]])

best_maybe_model = LinearRegression()
best_maybe_model.fit(X_best_maybe, y_train)

cv_best_maybe_model = cross_validate(
    best_maybe_model,
    X_best_maybe,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

In [72]:
print(np.sqrt(-sum(cv_full_model['test_score'])),
      np.sqrt(-sum(cv_interaction_model['test_score'])),
      np.sqrt(-sum(cv_poly_model['test_score'])),
      np.sqrt(-sum(cv_best_maybe_model['test_score'])))

1.1548944121374503 1.0429122087484146 0.9348946913871552 2.108284217349092


It seems like our `X_best_maybe` was in fact not the best, but it might be the best for only two variables in the data. Obviously, the other models also include the `residual sugar` and `density` variables in them as well, so the extra variables seem to cause the mean_squared_error to be lower for predicting `alcohol`.

If given enough time, I will make a script that will test any combination of 2 variables in our data set to see which two variables give the lowest RMSE.

The polynomial model that included interaction and none-interaction terms of degree two had the lowest RMSE at 0.935.

In [20]:
#####################
#
# Bonus Time
#
#######################

from itertools import combinations

original_vars = wine_data.columns.tolist()
original_vars.remove("alcohol")
original_vars.remove("type")

linear_list = []

for r in range(1, 3):
    for combo in combinations(original_vars, r):
        cv_model = cross_validate(
            LinearRegression(),
            X_train_reg[list(combo)],
            y_train,
            cv=5,
            scoring="neg_mean_squared_error"
        )
        score = np.sqrt(-sum(cv_model['test_score']))
        linear_list.append((combo, score))

sorted_list = sorted(linear_list, key=lambda x: x[1])

print(sorted_list[0])

(('density', 'type_num'), np.float64(1.8212508331148969))


It seems like Density and Type produce the smallest RMSE based on solely the two variables and no interaction terms! Type was just 0 for red and 1 for white. This reveals that the type of wine has a big impact on predicting the amount of alcohol!

In [21]:
inter_list = []

for r in range(1, 3):
    for combo in combinations(original_vars, r):
        X_train_interations = poly_interation.fit_transform(X_train_reg[list(combo)])
        cv_model = cross_validate(
            LinearRegression(),
            X_train_interations,
            y_train,
            cv=5,
            scoring="neg_mean_squared_error"
        )
        score = np.sqrt(-sum(cv_model['test_score']))
        inter_list.append((combo, score))

sorted_inter_list = sorted(inter_list, key=lambda x: x[1])

print(sorted_inter_list[0])

(('fixed acidity', 'density'), np.float64(1.7957911073192032))


It seems like Fixed Acidity and Density produced the lowest RMSE when interaction terms were introduced. I have no clue at what the easiest things to measure are for wine. So is this better? I do not know, but I do know it has a slightly lower RMSE than just Density and Quality without an interaction term! Type of Wine is probably the easiest thing to "measure" for the wine in our prediction.

## LASSO-ing a Model

Now we are going to attempt to do the problem using a LASSO model.

In [50]:
from sklearn import linear_model
from sklearn.linear_model import LassoCV

lasso_model = LassoCV(cv=5, random_state = 0) \
    .fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values,
         y_train.values)

In [51]:
lasso_model.alpha_

np.float64(0.0008255721135897033)

In [52]:
lasso_best = linear_model.Lasso(lasso_model.alpha_)
lasso_best.fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values,
         y_train.values)

print(lasso_best.intercept_, lasso_best.coef_)

10.489583092809903 [-2.01982765 -0.61833023  0.73504031  1.09611634  0.44392546]


I used the variables - pH, type_num, density, residual sugar, and fixed acidity - in my Lasso Model to try to find the best fit for the data in predicting alcohol content.

Lasso Model appears to be optimized at alpha = 0.0008256. This is slightly larger than 0, but that does produce a different intercept and coefficients than if alpha did equal 0.

### Ridge Regression

In [40]:
from sklearn.linear_model import RidgeCV

ridge_model_1 = RidgeCV(cv=5) \
    .fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values,
         y_train.values)

In [41]:
ridge_model_1.alpha_

np.float64(1.0)

In [42]:
ridge_best_1 = linear_model.Ridge(ridge_model_1.alpha_)
ridge_best_1.fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values,
         y_train.values)

print(ridge_best_1.intercept_, ridge_best_1.coef_)

10.489583092809903 [-2.02735275 -0.62161099  0.73905246  1.10315176  0.44657903]


The Ridge Regression was performed much in the same way that the Lasso Method was performed. The alpha came out to be 1.0. However, when I tried to run the RidgeCV command with `alphas =`a different value was returned.

 I'm not sure at this point if it has to do with the variables that I have chosen to produce these results, or if I am doing something incorrect.

In [55]:
ridge_model_2 = RidgeCV(cv=5, alphas = np.linspace(0, 5, 2000)) \
    .fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values,
         y_train.values)

print(ridge_model_2.alpha_)

3.0865432716358177


When running the Ridge Model with alphas = np.linspare(0,5,2000), the best alpha was 3.0865. Other combinations returned a different alpha. I'm not entirely sure what is happening in this model for different results to happen, or if it is because of the way that alphas/linspace are working in combination with each other.

In [46]:
ridge_best_2 = linear_model.Ridge(ridge_model_2.alpha_)
ridge_best_2.fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values,
         y_train.values)

print(ridge_best_2.intercept_, ridge_best_2.coef_)

10.489583092809903 [-2.02089538 -0.61880596  0.73610145  1.0974357   0.44496448]


When looking at the model intercept and coefficient, you can see that the intercepts are the same for both Ridge Models, but the 'slopes' are slightly different, which should lead to different predictions for each model.

### Elastic Net

Time for the final model we will be testing.

In [66]:
from sklearn.linear_model import ElasticNetCV


el_net_model = ElasticNetCV(cv=5, random_state=42) \
    .fit(X_train_reg[['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']].values, y_train.values)

# Print the chosen alpha and l1_ratio
print("Alpha:", el_net_model.alpha_)
print("Number of Iterations:", el_net_model.n_iter_)
print("L1 Ratio:", {el_net_model.l1_ratio_})

Alpha: 0.0016511442271794066
Number of Iterations: 44
L1 Ratio: {np.float64(0.5)}


The chosen alpha is 0.00165, with a L1 Ratio if 0.5. It took the program 44 iterations to choose this alpha. The L1 Ratio is the ration between the L1 (Lasso) and L2 (Ridge) penalties.

In [67]:
print(el_net_model.intercept_, el_net_model.coef_)

10.489583092809903 [-2.00664841 -0.61260558  0.72901816  1.08445235  0.44063112]


Again, all of the slopes look similar to the slopes of the other models with some slight differences.

I chose to do the same variables for each model to make sure that the code was producing similar models.

## Testing Our Models

It is now time to test our models. We will compare our chosen models based on RMSE and MAE.

In [89]:
from sklearn.metrics import mean_squared_error

# Standardize X_test_reg using the means and stds from the training data
X_test_reg_std = (X_test_reg - x_means) / x_stds

#LR - full model
y_pred_full = lr_full.predict(X_test_reg_std)
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred_full))
print("Linear Full Model RMSE:", rmse_full)

#Interation Model
y_pred_inter = inter_model.predict(poly_interation.fit_transform(X_test_reg_std))
rmse_inter = np.sqrt(mean_squared_error(y_test, y_pred_inter))
print("Interaction Model RMSE:", rmse_inter)

#Polynomial Model
y_pred_poly = poly_model.predict(poly_.fit_transform(X_test_reg_std))
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
print("Polynomial Model RMSE:", rmse_poly)

#Best (Maybe) Model
X_test_std_best = poly_interation.fit_transform(X_test_reg_std[["residual sugar", "density"]])
y_pred_best = best_maybe_model.predict(X_test_std_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
print("Best Model RMSE:", rmse_best)

vars = ['density', 'type_num', 'fixed acidity', 'residual sugar', 'pH']

#Lasso Model
y_pred_lasso = lasso_best.predict(X_test_reg_std[vars].values)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
print("Lasso Model RMSE:", rmse_lasso)

#Ridge Model 1
y_pred_ridge_1 = ridge_best_1.predict(X_test_reg_std[vars].values)
rmse_ridge_1 = np.sqrt(mean_squared_error(y_test, y_pred_ridge_1))
print("Ridge Model #1 RMSE:", rmse_ridge_1)

#Ridge Model 2
y_pred_ridge_2 = ridge_best_2.predict(X_test_reg_std[vars].values)
rmse_ridge_2 = np.sqrt(mean_squared_error(y_test, y_pred_ridge_2))
print("Ridge Model #2 RMSE:", rmse_ridge_2)

#Elastic Net Model
y_pred_el_net = el_net_model.predict(X_test_reg_std[vars].values)
rmse_el_net = np.sqrt(mean_squared_error(y_test, y_pred_el_net))
print("Elastic Net Model RMSE:", rmse_el_net)

Linear Full Model RMSE: 0.4653460972974002
Interaction Model RMSE: 0.42811561511364415
Polynomial Model RMSE: 0.44330994006708047
Best Model RMSE: 0.8866110785827737
Lasso Model RMSE: 0.49555418442516763
Ridge Model #1 RMSE: 0.49520079737246253
Ridge Model #2 RMSE: 0.49549099854799894
Elastic Net Model RMSE: 0.4962144039472083


Based on the RMSE, all of the models were very similar (except the best maybe model), but the interaction model produced the lowest RMSE.

In [96]:
from sklearn.metrics import mean_absolute_error

#LR - full model
mae_full = mean_absolute_error(y_test, y_pred_full)
print("Linear Full Model MAE:", mae_full)

#Interation Model
mae_inter = mean_absolute_error(y_test, y_pred_inter)
print("Interaction Model MAE:", mae_inter)

#Polynomial Model
mae_poly = mean_absolute_error(y_test, y_pred_poly)
print("Polynomial Model MAE:", mae_poly)

#Best (Maybe) Model
mae_best = mean_absolute_error(y_test, y_pred_best)
print("Best Model MAE:", mae_best)

#Lasso Model
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
print("Lasso Model MAE:", mae_lasso)

#Ridge Model 1
mae_ridge_1 = mean_absolute_error(y_test, y_pred_ridge_1)
print("Ridge Model #1 MAE:", mae_ridge_1)

#Ridge Model 2
mae_ridge_2 = mean_absolute_error(y_test, y_pred_ridge_2)
print("Ridge Model #2 MAE:", mae_ridge_2)

#Elastic Net Model
mae_el_net = mean_absolute_error(y_test, y_pred_el_net)
print("Elastic Net Model MAE:", mae_el_net)

Linear Full Model MAE: 0.35078764817362956
Interaction Model MAE: 0.3093886998518352
Polynomial Model MAE: 0.3069597401526713
Best Model MAE: 0.6842414993892894
Lasso Model MAE: 0.3646548406458771
Ridge Model #1 MAE: 0.3642996867151421
Ridge Model #2 MAE: 0.36464270645238595
Elastic Net Model MAE: 0.3654317506061257


Again, the "Best Model Maybe" produced the worst MAE. This is to be expected compared to the interaction and polynomial model, as those models contain all of the variables, while the Best Model Maybe only contains 2 different variables.

This time, the polynomial model had a slightly lower MAE compared to the other models. The Lasso, two Ridge models, and the Elastic Net model were all very very similar.

# Classification Task

Now, we are going to see if we can create an algorithm to predict the type of wine based on some variables about the wine.

In [120]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter = 1000)

X = wine_data.drop(columns=["type", "type_num"])
y = wine_data["type_num"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = wine_data['type'])

cv_log_full= cross_validate(
    log_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_1= cross_validate(
    log_model,
    X_train[["density", "pH"]],
    y_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_2= cross_validate(
    log_model,
    X_train[["density", "residual sugar"]],
    y_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_3= cross_validate(
    log_model,
    X_train[["density", "residual sugar", "pH"]],
    y_train,
    cv=5,
    scoring="neg_log_loss"
)

cv_log_4= cross_validate(
    log_model,
    X_train[['density', 'fixed acidity', 'residual sugar', 'pH']],
    y_train,
    cv=5,
    scoring="neg_log_loss"
)

print("Mean CV Log Loss:", -cv_log_full["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_1["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_2["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_3["test_score"].mean())
print("Mean CV Log Loss:", -cv_log_4["test_score"].mean())

Mean CV Log Loss: 0.0640445574558753
Mean CV Log Loss: 0.5024110033225874
Mean CV Log Loss: 0.46905917489231336
Mean CV Log Loss: 0.43806138734763067
Mean CV Log Loss: 0.25181270318210924


Not suprisingly, the more variables we include in the predictions, the lower the negative log loss is. This is when I would look at different combinations of 2 or 3 variables to see which combo produces the lowest log loss.

### Testing

In [131]:
log_full = LogisticRegression(max_iter=1000).fit(X_train, y_train)

log_1 = LogisticRegression(max_iter=1000).fit(
    X_train[["density", "pH"]], y_train)

log_2 = LogisticRegression(max_iter=1000).fit(
    X_train[["density", "residual sugar"]], y_train)

log_3 = LogisticRegression(max_iter=1000).fit(
    X_train[["density", "residual sugar", "pH"]], y_train)

log_4 = LogisticRegression(max_iter=1000).fit(
    X_train[['density', 'fixed acidity', 'residual sugar', 'pH']], y_train)

y_pred_full = log_full.predict(X_test)
y_pred_1 = log_1.predict(X_test[["density", "pH"]])
y_pred_2 = log_2.predict(X_test[["density", "residual sugar"]])
y_pred_3 = log_3.predict(X_test[["density", "residual sugar", "pH"]])
y_pred_4 = log_4.predict(X_test[['density', 'fixed acidity', 'residual sugar', 'pH']])

In [133]:
from sklearn.metrics import log_loss, accuracy_score

print("Full Model Log Loss:", log_loss(y_test, y_pred_full))
print("Full Model Accuracy:", accuracy_score(y_test, y_pred_full))

Full Model Log Loss: 0.5822436316703542
Full Model Accuracy: 0.9838461538461538


In [134]:
print("Full Model Log Loss:", log_loss(y_test, y_pred_1))
print("Full Model Accuracy:", accuracy_score(y_test, y_pred_1))

Full Model Log Loss: 8.900009798389698
Full Model Accuracy: 0.7530769230769231


In [136]:
print("Full Model Log Loss:", log_loss(y_test, y_pred_2))
print("Full Model Accuracy:", accuracy_score(y_test, y_pred_2))

Full Model Log Loss: 8.8722839111673
Full Model Accuracy: 0.7538461538461538


In [138]:
print("Full Model Log Loss:", log_loss(y_test, y_pred_3))
print("Full Model Accuracy:", accuracy_score(y_test, y_pred_3))

Full Model Log Loss: 8.650476813388117
Full Model Accuracy: 0.76


In [139]:
print("Full Model Log Loss:", log_loss(y_test, y_pred_4))
print("Full Model Accuracy:", accuracy_score(y_test, y_pred_4))

Full Model Log Loss: 4.2420607450268655
Full Model Accuracy: 0.8823076923076923


The full model had by far the lowest log loss, and the best accuracy score. My 2nd, 3rd, and 4th model were all pretty similar to one another.

The accuracy and log loss seem to improve with more variables included. I have no idea how easy/hard to get thses measurements. If I knew how easy it would be to record various vales for your wine, then I could look and compare the log loss and accuracy of only using easy to measure variables.